# SkyRun Inc. Customer Retention & Cohort Analysis



| Skill | Where |
|---|---|
| KPIs, questions, assumptions | Section 1 |
| pandas: loading, inspecting, cleaning, filtering, grouping, merging, pivoting | Sections 2, 3, 5, 6, 7 |
| matplotlib & seaborn: Visualisations | Section 4 |
| Cohort analysis and retention heatmaps | Section 5 |
| Drop-off, purchase intervals, churn | Section 6 |
| Segmentation, RFM, K-Means (unsupervised ML) | Sections 7 and 8 |
| Customer lifetime value (CLV) inputs | Section 9 |
| Recommendations and KPIs | Section 10 |

# 1. Business Understanding

## The situation
SkyRun is an e-commerce company. It wins lots of new customers, but **too few come back to buy again**, and retention has been getting worse. Acquiring a customer costs money (marketing, discounts), so a customer who buys once and disappears often never pays back their acquisition cost.

## Key terms 

| Term | Plain-English meaning | How its measured here |
|---|---|---|
| **Cohort** | A group of customers who share a starting point | All customers whose **first purchase** was in the same month |
| **Retention rate** | Of the customers who joined in a month, what % are still buying later? | Customers active in month *n* / cohort size |
| **Repeat purchase rate** | % of customers who bought **more than once** | Customers with 2+ orders / all customers |
| **Churn** | A customer who has stopped buying | No purchase for longer than a chosen number of days |
| **AOV** (average order value) | How much one order is worth | Total revenue / number of orders |
| **CLV** (customer lifetime value) | Total value a customer brings over their relationship with us | Revenue per customer over time (cohort based) |
| **CAC** (customer acquisition cost) | What it costs to win one customer | Marketing spend / new customers. **We have no spend data, so we cannot compute it** (we will say so honestly) |
| **RFM** | Recency, Frequency, Monetary: three numbers that describe a customer's behaviour | Days since last order, number of orders, total spend |

## Research Problem

| Objective | The question we will answer | Section |
|---|---|---|
| 1. Measure retention | What % of each monthly cohort comes back in month 1, 2, 3...? | 5 |
| 2. Segment customers | Do customer groups (cohort month, behaviour, first category, country) differ? | 7, 8 |
| 3. Find drop-off points | After which order, and after how many days of silence, do customers disappear? | 6 |
| 4. Support CLV | How much revenue does an average customer bring in their first months? | 9 |
| 5. Recommendations | What should SkyRun *do*, and how will it know it worked? | 10 |

## Libraries

In [ ]:
# Libraries
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import FuncFormatter, PercentFormatter

# Machine learning tools (used in Section 8)
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

import warnings
warnings.filterwarnings("ignore")

# Show all columns when we print a table, and give every seaborn chart a clean look
pd.set_option("display.max_columns", 30)
sns.set_theme(style="whitegrid")

# 2. Data Ingestion & Understanding

In [ ]:
df = pd.read_csv("/home/eoa/Documents/Amdari Projects/WEEK 3/Skyrun dataset.csv")
df.shape           

In [ ]:
# head() shows the first 5 rows 
df.head()

In [ ]:
df.info()

**Reading `info()`:** `CustomerID` has fewer non-null values than the other columns (missing customers) and `InvoiceDate` is stored as plain text (`str`), not as a date. Both need fixing.

### Data dictionary (what each column means)
| Column | Meaning | Grain / note |
|---|---|---|
| `InvoiceNo` | Order number | One order can have **many rows** |
| `InvoiceDate` | When the order was placed | Text for now |
| `CustomerID` | Who bought | Missing for some rows |
| `StockCode` | Product code | |
| `Description` | In this file it is a **product category** (Books, Toys...) | Not a product name |
| `Quantity` | Units bought on this line | |
| `UnitPrice` | Price of one unit | |
| `Country` | Country on the line | See the problem we find below |

So **one row = one product line inside an order**. 

In [ ]:
# describe() gives count, mean, min, quartiles and max for the number columns
df.describe()

In [ ]:
# Missing values
print(df.isnull().sum())

# Same thing as a percentage 
print()
print((df.isnull().mean() * 100).round(1))

In [ ]:
# Convert the text column to real dates
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])

print("First transaction:", df["InvoiceDate"].min())
print("Last transaction :", df["InvoiceDate"].max())

# How many rows per month? (strftime turns a date into text such as '2025-03')
df["InvoiceDate"].dt.strftime("%Y-%m").value_counts().sort_index()

In [ ]:
# Counting categories: value_counts() and nunique()
print(df["Description"].value_counts())
print()
print("Number of different countries :", df["Country"].nunique())
print("Number of different invoices  :", df["InvoiceNo"].nunique())
print("Number of different customers :", df["CustomerID"].nunique())

In [ ]:
# 1) Does an invoice belong to ONE country?
print("Average number of countries inside ONE invoice:", round(df.groupby("InvoiceNo")["Country"].nunique().mean(), 1))

# 2) Does a customer belong to ONE country?
print("Average number of countries per customer:", round(df.groupby("CustomerID")["Country"].nunique().mean(), 1))

# 3) Does one StockCode always have the same category?
print("Average categories per StockCode:", round(df.groupby("StockCode")["Description"].nunique().mean(), 1))

# 4) Are there invoices that start with a letter?
print("Invoices that do not start with a digit:", (~df["InvoiceNo"].str.match("[0-9]")).sum(), "rows")

## What the checks tell us 

1. **About 25% of the rows have no `CustomerID`.** We cannot follow a customer we cannot identify, so these rows cannot be used for retention.
2. **Invoices starting with `C` are cancellations** and `A` are adjustments. They are not real purchases.
3. **`Description` is the product category**, and a `StockCode` maps to many categories, so we only use the category.
4. **Dates run from 1 Dec 2024 to 9 Dec 2025.** December 2025 is an **incomplete month** (only 9 days).

# 3. Data Cleaning & Preparation

In [ ]:
# A log of exactly what we removed at each step
cleaning_log = []
def log(step, data):
    cleaning_log.append({"Step": step, "Rows left": len(data)})

log("Raw data", df)

# Keep a copy of the raw data so we can measure what share of revenue each step removes
df["Revenue"] = df["Quantity"] * df["UnitPrice"]     
df_raw = df.copy()

**create `Revenue`.** One line's revenue = units x price. 

In [ ]:
# cancellations and adjustments (InvoiceNo starts with C or A)
is_cancel = df["InvoiceNo"].str.match("[CA]")       # True/False for every row
print(is_cancel.sum(), "rows are cancellations / adjustments")
print("Share of revenue:", round(df.loc[is_cancel, "Revenue"].sum() / df["Revenue"].sum() * 100, 1), "%")

df[is_cancel].head(3)

In [ ]:
# ~is_cancel means NOT is_cancel. 
df = df[~is_cancel]
log("Removed cancellations/adjustments", df)

> **Decision (log #1):** remove cancelled orders.

In [ ]:
# missing CustomerID
missing_share = df[df["CustomerID"].isnull()]["Revenue"].sum() / df["Revenue"].sum()
print("Revenue sitting in rows with no customer:", round(missing_share * 100, 1), "%")

# droping rows that are missing in the named column only
df = df.dropna(subset=["CustomerID"])

# IDs were floats (17850.0) only because of the missing values
df["CustomerID"] = df["CustomerID"].astype(int)
log("Removed rows with no CustomerID", df)

> **Decision (log #2):** drop rows with no customer. We *cannot* build a cohort for an anonymous buyer. The cost is real: that is a quarter of the revenue, so **totals in this notebook describe identified customers, not the whole company.** 

In [ ]:
# impossible values and duplicates
print("Rows with quantity <= 0 or price <= 0:", ((df["Quantity"] <= 0) | (df["UnitPrice"] <= 0)).sum())
print("Fully duplicated rows:", df.duplicated().sum())

df = df.drop_duplicates()
log("Removed duplicates", df)

In [ ]:
# date helper columns we will need for monthly analysis
df["OrderMonth"] = df["InvoiceDate"].dt.strftime("%Y-%m")        # text like 2025-03, sorts correctly
pd.DataFrame(cleaning_log)

In [ ]:
orders = df.groupby("InvoiceNo").agg(
    CustomerID = ("CustomerID", "first"),       # an order has one customer, take the first value
    OrderDate  = ("InvoiceDate", "min"),        # earliest time on the invoice
    Revenue    = ("Revenue", "sum"),            # add up the lines
    Items      = ("Quantity", "sum"),
).reset_index()                                  # turn InvoiceNo back into a normal column

orders["OrderMonth"] = orders["OrderDate"].dt.strftime("%Y-%m")
# MonthNum counts months as one number (year*12+month) so we can subtract months later
orders["MonthNum"] = orders["OrderDate"].dt.year * 12 + orders["OrderDate"].dt.month

print(orders.shape)
orders.head()

In [ ]:
# CUSTOMER table: one row per customer
customers = orders.groupby("CustomerID").agg(
    Orders       = ("InvoiceNo", "count"),
    TotalRevenue = ("Revenue", "sum"),
    FirstOrder   = ("OrderDate", "min"),
    LastOrder    = ("OrderDate", "max"),
).reset_index()

print(customers.shape)
customers.head()

**Check:** our customer table has about 4,300 rows, so each row is a person. We now have clean line, order and customer tables.

In [ ]:
# First we need something to draw: revenue, orders and active customers per month
monthly = orders.groupby("OrderMonth").agg(
    Orders          = ("InvoiceNo", "count"),
    ActiveCustomers = ("CustomerID", "nunique"),
    Revenue         = ("Revenue", "sum"),
).reset_index()
monthly

**It works, but it is not communicating.** No title, unreadable numbers (`1e8`), crowded labels. Let us fix it step by step.

In [ ]:
# LINE CHART
fig, ax = plt.subplots(figsize=(10, 4))

ax.plot(monthly["OrderMonth"], monthly["Revenue"], marker="o", color="#1f77b4", linewidth=2)

# Words: the title should state the message
ax.set_title("Monthly revenue grew strongly from September 2025", fontsize=14, weight="bold")
ax.set_xlabel("Order month")
ax.set_ylabel("Revenue (millions)")

# Turn 100,000,000 into 100M using a small formatting function
ax.yaxis.set_major_formatter(FuncFormatter(lambda value, pos: f"{value/1e6:.0f}M"))

# Rotate month labels so they do not overlap
ax.tick_params(axis="x", rotation=45)

# Annotate the peak so the reader does not have to hunt for it
peak = monthly.loc[monthly["Revenue"].idxmax()]
ax.annotate(f"Peak: {peak['OrderMonth']}", xy=(peak["OrderMonth"], peak["Revenue"]),
            xytext=(-110, -5), textcoords="offset points", arrowprops=dict(arrowstyle="->"))

# Flag that the last month is incomplete (only 9 days of December 2025)
last = monthly.iloc[-1]
ax.annotate("partial month", xy=(last["OrderMonth"], last["Revenue"]), xytext=(-5, 14),
            textcoords="offset points", ha="center", fontsize=9, color="grey")

ax.grid(axis="x", visible=False)             # remove the vertical lines
ax.grid(axis="y", linestyle="--", alpha=0.5)
fig.tight_layout()                           # stops labels from being cut off
plt.show()

**Reading the chart:** revenue is fairly flat through the summer and climbs sharply from September. The last point drops only because December is incomplete, not because customers left. *Seeing this is exactly why we looked at the dates in Section 2.*

In [ ]:
# BAR CHART: which category earns most? 
cat_revenue = df.groupby("Description")["Revenue"].sum().sort_values()    # sort so the biggest ends on top

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(cat_revenue.index, cat_revenue.values / 1e6, color="#4c72b0")     # barh = horizontal bars
ax.bar_label(ax.containers[0], fmt="%.0f", padding=3)                      # print the value at the end of each bar
ax.set_title("Revenue by product category (millions)", fontsize=14, weight="bold")
ax.set_xlabel("Revenue (millions)")
fig.tight_layout()
plt.show()

**Reading it:** the ten categories earn almost the same revenue, so SkyRun is not dependent on one category. 

In [ ]:
# HISTOGRAM: how is order value spread?
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(orders["Revenue"], bins=40, color="#55a868", edgecolor="white")

# Mark the average and the median with vertical lines
ax.axvline(orders["Revenue"].mean(),   color="red",    linestyle="--", label="Mean")
ax.axvline(orders["Revenue"].median(), color="black",  linestyle="--", label="Median")

ax.set_title("Most orders are worth a modest amount; a few are very large", fontsize=13, weight="bold")
ax.set_xlabel("Order value")
ax.set_ylabel("Number of orders")
ax.legend()
plt.show()

**mean is bigger than the median**, a few big values are pulling the average up (a **right-skewed** distribution). Skew matters later: it is the reason we take a **log** before clustering.

In [ ]:
# BOX PLOT with seaborn: compare the spread of order value across groups.
# sample() takes a random subset so the chart draws quickly
sample = df.sample(50000, random_state=1)

fig, ax = plt.subplots(figsize=(10, 4))
sns.boxplot(data=sample, x="Description", y="Revenue", color="#8172b3", ax=ax)
ax.set_title("Line revenue is spread almost identically across categories", fontsize=13, weight="bold")
ax.set_xlabel("")
ax.tick_params(axis="x", rotation=45)
plt.show()

In [ ]:
# SCATTER PLOT: is spending related to the number of orders?
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(customers["Orders"], customers["TotalRevenue"] / 1e6, alpha=0.3, s=15)   # alpha = transparency
ax.set_xscale("log")                       # log scale spreads out the crowded left side
ax.set_title("Customers who order more spend more", fontsize=13, weight="bold")
ax.set_xlabel("Number of orders (log scale)")
ax.set_ylabel("Total revenue (millions)")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))      

metrics = [("Orders", "#4c72b0"), ("ActiveCustomers", "#55a868"), ("Revenue", "#c44e52")]
for ax, (col, colour) in zip(axes, metrics):            # loop: one chart per metric
    ax.plot(monthly["OrderMonth"], monthly[col], color=colour, marker="o")
    ax.set_title(col, weight="bold")
    ax.tick_params(axis="x", rotation=90)

fig.suptitle("SkyRun monthly activity (last month is incomplete)", fontsize=14, weight="bold")
fig.tight_layout()
plt.show()

In [ ]:
grid = pd.pivot_table(df, index="Description", columns="OrderMonth", values="Revenue", aggfunc="sum") / 1e6

fig, ax = plt.subplots(figsize=(12, 5))
sns.heatmap(grid, cmap="YlGnBu", linewidths=0.5, cbar_kws={"label": "Revenue (millions)"}, ax=ax)
ax.set_title("Revenue by category and month: every category rises together from September", weight="bold")
ax.set_xlabel(""); ax.set_ylabel("")
ax.grid(False)                               # heatmaps look cleaner without grid lines
plt.show()

# 5. Cohort Construction & Retention Analysis
**Business question:** *Of the customers SkyRun wins in a month, how many are still buying 1, 2, 3... months later?*

## TGoal
1. Find each customer's **first purchase month**  that is their **cohort**.
2. For every later order, work out **how many months after the first purchase** it happened - the **cohort index** (0 = the first month).
3. Count the **distinct customers who bought** in each (cohort, index) cell.
4. Divide by the cohort's size (index 0) to get **retention %**.


In [ ]:
# give every customer their cohort (first order month)
customers["CohortMonth"] = customers["FirstOrder"].dt.strftime("%Y-%m")
customers["CohortNum"]   = customers["FirstOrder"].dt.year * 12 + customers["FirstOrder"].dt.month

# attach the cohort to every order with merge 
orders = orders.merge(customers[["CustomerID", "CohortMonth", "CohortNum"]], on="CustomerID", how="left")

# Step 3: months since the first purchase
orders["CohortIndex"] = orders["MonthNum"] - orders["CohortNum"]
orders[["InvoiceNo", "CustomerID", "OrderMonth", "CohortMonth", "CohortIndex"]].head()

> **Decision (log #4) - the analysis window.** Our last transaction is 9 December 2025. Counting December would make every cohort look as if it collapsed in its last column, because the month is only 9 days long. **We analyse complete months only: December 2024 to November 2025.** Customers whose first order was in December 2025 are not part of the retention table (they are still in the RFM and CLV work).

> **Decision (log #5) - the December 2024 cohort is special.** The data starts on 1 December 2024, so everyone who bought in that first month looks "new" even though many were existing customers. This is called **left-censoring** (we cannot see their history). That is why the first cohort is much bigger than the others. **We keep it in the table but exclude it when we compute averages.**

In [ ]:
# Keep only complete months
window_end = pd.Timestamp("2025-12-01")
orders_w = orders[orders["OrderDate"] < window_end].copy()
print("Orders in the analysis window:", len(orders_w), "of", len(orders))

In [ ]:
# Count DISTINCT customers who ordered in each (cohort, months since first order)
cohort_data = orders_w.groupby(["CohortMonth", "CohortIndex"])["CustomerID"].nunique().reset_index()

# pivot: cohorts down the side, months-since-first-purchase across the top
cohort_counts = cohort_data.pivot(index="CohortMonth", columns="CohortIndex", values="CustomerID")

# The first column (index 0) is the cohort size = everyone who joined that month
cohort_size = cohort_counts[0]

# Retention % = customers active in month n / cohort size
retention = cohort_counts.divide(cohort_size, axis=0) * 100
retention.round(1)

In [ ]:
# THE RETENTION HEATMAP 
fig, ax = plt.subplots(figsize=(13, 7))
sns.heatmap(retention, annot=True, fmt=".0f", cmap="Blues", vmin=0, vmax=60,
            linewidths=0.5, cbar_kws={"label": "% of cohort still buying"}, ax=ax)
ax.set_title("Cohort retention: a big drop after the first month, then a steady plateau", fontsize=14, weight="bold")
ax.set_xlabel("Months since first purchase")
ax.set_ylabel("Cohort (month of first purchase)")
ax.grid(False)
plt.show()

In [ ]:
# Average retention curve. Exclude 2024-12 (left-censored). mean() skips the blanks automatically.
avg_retention = retention.drop(index="2024-12").mean()

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(avg_retention.index, avg_retention.values, marker="o", linewidth=2, label="Average of 2025 cohorts")
ax.plot(retention.columns, retention.loc["2024-12"], marker="s", linestyle="--", color="grey", label="Dec 2024 cohort (includes existing customers)")

# Print the value above each point (zip pairs x with y)
for x, y in zip(avg_retention.index[1:], avg_retention.values[1:]):
    ax.annotate(f"{y:.0f}%", (x, y), textcoords="offset points", xytext=(0, 8), ha="center", fontsize=9)

ax.yaxis.set_major_formatter(PercentFormatter())
ax.set_ylim(0, 105)
ax.set_title("The average retention curve", fontsize=14, weight="bold")
ax.set_xlabel("Months since first purchase")
ax.set_ylabel("% of cohort active")
ax.legend()
plt.show()

In [ ]:
# How big is each cohort? (acquisition volume) and how good was month-1 retention for each?
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

# Left: cohort sizes. Highlight the first bar because it is not a true "new customer" cohort.
colours = ["lightgrey" if m == "2024-12" else "#4c72b0" for m in cohort_size.index]
axes[0].bar(cohort_size.index, cohort_size.values, color=colours)
axes[0].set_title("New customers per month (grey = not a true cohort)", weight="bold")
axes[0].tick_params(axis="x", rotation=60)
axes[0].set_ylabel("Customers")

# Right: month-1 retention by cohort, with the 2025 average as a dashed line
month1 = retention[1].dropna()
colours = ["lightgrey" if m == "2024-12" else "#55a868" for m in month1.index]
axes[1].bar(month1.index, month1.values, color=colours)
axes[1].bar_label(axes[1].containers[0], fmt="%.0f%%", padding=2)
axes[1].axhline(month1.drop("2024-12").mean(), color="red", linestyle="--", label="2025 average")
axes[1].set_title("Month-1 retention by cohort", weight="bold")
axes[1].tick_params(axis="x", rotation=60)
axes[1].yaxis.set_major_formatter(PercentFormatter())
axes[1].legend()

fig.tight_layout()
plt.show()

### Results
- **Month-1 retention averages about 20% for the 2025 cohorts**, so roughly **4 in 5 new customers do not buy again in the calendar month after their first purchase**. That is the big drop-off.
- After month 1 the curve does **not keep falling**. It sits around 22-25% and then rises. A real business usually shows steady decay, so this flat shape is a reminder the data is synthetic. Do not over-interpret small wiggles.
- **Cohorts differ a little.** Month-1 retention was 22% for January, fell to its lowest point of **15% for March**, stayed between 17% and 21% through the summer, and recovered to **23-24% for the September and October cohorts**. So we see a **mid-year dip and an autumn recovery, not a steady decline**. (The brief mentions 18 months of decline; we only have 12 months of data, so we cannot confirm that claim.)
- **Diagonal effect:** look at the late columns of the older rows (months 8-10). Average retention climbs to about 28%, 32% and 37% in months 8, 9 and 10. That is **not** customers becoming more loyal; those columns are September-November 2025, when the whole business surged (Section 4). A diagonal or column that brightens together is a **calendar effect**, not a cohort effect.
- **Acquisition volume** fell from 417 new customers in January to 169 in August, then recovered to about 300-360 per month in September-November. Retention is only half the story: SkyRun also needs enough new customers coming in.

In [ ]:
# New vs returning customers each month: who is driving activity?
orders_w["CustomerType"] = np.where(orders_w["CohortIndex"] == 0, "New", "Returning")   # np.where = if/else for columns

mix = orders_w.groupby(["OrderMonth", "CustomerType"])["CustomerID"].nunique().unstack()   # unstack: move CustomerType into columns

fig, ax = plt.subplots(figsize=(11, 4.5))
mix.plot(kind="bar", stacked=True, color=["#c44e52", "#4c72b0"][::-1], ax=ax)   # pandas wraps matplotlib: quick bar charts
ax.set_title("Active customers each month: new vs returning", fontsize=14, weight="bold")
ax.set_xlabel(""); ax.set_ylabel("Active customers")
ax.tick_params(axis="x", rotation=45)
plt.show()

**How to read it:** each bar is the number of customers buying that month. The "Returning" part is the loyal base; the "New" part is fresh acquisition. A healthy business grows the blue (returning) part over time. (December 2024 is entirely "New" only because the data begins there.)

# 6. Customer Drop-Off Analysis
**Business question:** *When do customers disappear - after how many orders, and after how many days of silence?*

Cohort tables show *calendar months*. To act on churn we also need **order-by-order** and **days-between-orders** views.

In [ ]:
# Put every customer's orders in time order, then number them 1, 2, 3...
orders = orders.sort_values(["CustomerID", "OrderDate"])

# cumcount() counts rows inside each group starting at 0, so +1 gives the order number
orders["OrderNumber"] = orders.groupby("CustomerID").cumcount() + 1

# diff() = this value minus the previous one INSIDE each customer days since their previous order
orders["DaysSincePrev"] = orders.groupby("CustomerID")["OrderDate"].diff().dt.days
orders[["CustomerID", "OrderDate", "OrderNumber", "DaysSincePrev"]].head(8)

In [ ]:
# How many orders do customers place? A look at the distribution
order_counts = customers["Orders"].value_counts().sort_index()
print("Share of customers with exactly 1 order:", round((customers["Orders"] == 1).mean() * 100, 1), "%")
print("Share with 2 or more orders (repeat purchase rate):", round((customers["Orders"] >= 2).mean() * 100, 1), "%")
print("Median orders per customer:", customers["Orders"].median())

In [ ]:
# The "orders ladder": how many customers reach order 1, 2, 3...?
reached = orders.groupby("OrderNumber")["CustomerID"].nunique().head(10)       # customers who placed at least n orders
pct_reached = reached / reached.iloc[0] * 100                                 # % of ALL customers
pct_continue = reached / reached.shift(1) * 100                               # % of those at order n-1 who made order n

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

axes[0].bar(pct_reached.index, pct_reached.values, color="#4c72b0")
axes[0].bar_label(axes[0].containers[0], fmt="%.0f%%", padding=2)
axes[0].set_title("% of all customers who reach each order", weight="bold")
axes[0].set_xlabel("Order number"); axes[0].set_xticks(pct_reached.index)

axes[1].bar(pct_continue.index[1:], pct_continue.values[1:], color="#dd8452")
axes[1].bar_label(axes[1].containers[0], fmt="%.0f%%", padding=2)
axes[1].set_title("% who go on to the NEXT order (continuation rate)", weight="bold")
axes[1].set_xlabel("Order placed (as a % of customers who placed the one before)"); axes[1].set_xticks(pct_continue.index[1:])

for ax in axes: ax.yaxis.set_major_formatter(PercentFormatter()); ax.set_ylim(0, 105)
fig.tight_layout()
plt.show()

**Reading the two charts**
- **Left:** 100% of customers place a first order, **66% place a second**, **46% a third** and **26% a fifth**. The largest single loss is **between the first and second order**: 34% of customers never come back at all.
- **Right:** the share who go on to the *next* order **rises** the further customers get: 66% after order 1, 71% after order 2, 75% after order 3, 85% after order 9. **The more a customer has bought, the more likely they are to buy again.** A habit forms. So the business job is to **get customers over the early hurdles (orders 2 and 3)**.
- Why does this look so much better than the 20% month-1 retention? Month-1 retention only counts customers who return **in the very next calendar month**. Many return later, which the next two charts show.

In [ ]:
# Time between orders: the "rhythm" of a customer
gaps = orders["DaysSincePrev"].dropna()
print(gaps.describe().round(1))
print()
print("Percentiles of days between orders:")
print(gaps.quantile([0.5, 0.75, 0.9, 0.95]))

fig, ax = plt.subplots(figsize=(9, 4))
sns.histplot(gaps, bins=60, color="#55a868", ax=ax)
ax.axvline(gaps.median(), color="black", linestyle="--", label=f"Median: {gaps.median():.0f} days")
ax.axvline(gaps.quantile(0.9), color="red", linestyle="--", label=f"90th percentile: {gaps.quantile(0.9):.0f} days")
ax.set_title("Days between consecutive orders", fontsize=13, weight="bold")
ax.set_xlabel("Days since previous order"); ax.set_ylabel("Number of orders")
ax.legend()
plt.show()

> **Decision (log #7) - how do we define "churned"?** **A customer who has been silent longer than 90% of normal gaps between orders is behaving abnormally**, so we set the churn threshold at the **90th percentile of the purchase gap**. (If the threshold were too short we would label active customers as lost and waste discounts on them; too long and we would react too late.) Always report the threshold you used.

In [ ]:
# Apply the definition. "snapshot" = the day after our last data, the moment we are judging from
snapshot = orders["OrderDate"].max() + pd.Timedelta(days=1)
churn_days = int(gaps.quantile(0.9))

customers["Recency"] = (snapshot - customers["LastOrder"]).dt.days      # days since their last order
customers["Churned"] = customers["Recency"] > churn_days                # True / False

print("Churn threshold:", churn_days, "days")
print("Share of customers flagged as churned:", round(customers["Churned"].mean() * 100, 1), "%")

In [ ]:
# How quickly do NEW customers place their second order?
# Add each customer's "days to second order" to the customer table (a left merge keeps one-time buyers as NaN)
second = orders[orders["OrderNumber"] == 2][["CustomerID", "DaysSincePrev"]].rename(columns={"DaysSincePrev": "DaysToSecond"})
customers = customers.merge(second, on="CustomerID", how="left")

# Fairness rule: only judge customers who have had at least 180 days to come back,
# and skip the 2024-12 group because we do not know when they truly started
cutoff = snapshot - pd.Timedelta(days=180)
mature = customers[(customers["FirstOrder"] >= "2025-01-01") & (customers["FirstOrder"] <= cutoff)]
print("Customers in this analysis:", len(mature))
print("Median days to second order (these customers, those who reordered):", mature["DaysToSecond"].median())
print("Median days to second order (ALL customers, biased low):", customers["DaysToSecond"].median())

days = list(range(0, 181))
cum_pct = [(mature["DaysToSecond"] <= d).mean() * 100 for d in days]    # NaN <= d is False, so one-time buyers count as "not yet"

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(days, cum_pct, linewidth=2)
for d in [30, 60, 90, 180]:
    ax.scatter(d, cum_pct[d], color="red", zorder=3)
    ax.annotate(f"{cum_pct[d]:.0f}% by day {d}", (d, cum_pct[d]), textcoords="offset points", xytext=(-10, 8), ha="right", fontsize=9)
ax.yaxis.set_major_formatter(PercentFormatter())
ax.set_title("What % of new customers have placed a second order by day X?", fontsize=13, weight="bold")
ax.set_xlabel("Days since first order"); ax.set_ylabel("% of customers")
plt.show()

**Reading the purchase-interval chart:** half of all repeat orders happen within **21 days** of the previous one, but there is a long tail. Only 10% of gaps are longer than **102 days**, which is why we use 102 days as the churn line. With this definition **31% of customers are currently churned.**

**Reading the second-order curve** (customers whose first order was between 1 January and 13 June 2025, so each has had at least 180 days to return; 1,945 customers):
- only **18% place a second order within 30 days**,
- **33% within 60 days**, **44% within 90 days** and **64% within 180 days**.

So most second orders arrive **between day 30 and day 180**, not in the first month. The first 30 days are a **quiet period where the business currently loses touch with a new customer**: the best window to remind, welcome and incentivise. Among these customers the **median second order comes about 71 days after the first**. (A figure computed on *all* customers gives about 50 days, but that is too low: recent customers who have not had time to reorder yet are missing from it, so we use the fair group.)

In [ ]:
# Churn by behaviour group. pd.cut puts a number into labelled bins.
customers["OrderGroup"] = pd.cut(customers["Orders"], bins=[0, 1, 3, 6, 10000],
                                 labels=["1 order", "2-3 orders", "4-6 orders", "7+ orders"])

churn_by_group = customers.groupby("OrderGroup", observed=True)["Churned"].mean() * 100
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(churn_by_group.index.astype(str), churn_by_group.values, color="#c44e52")
ax.bar_label(ax.containers[0], fmt="%.0f%%", padding=2)
ax.yaxis.set_major_formatter(PercentFormatter())
ax.set_title("More orders = much less churn", fontsize=13, weight="bold")
ax.set_ylabel("% of customers churned")
plt.show()

**Reading it:** a customer's number of orders is a strong signal of loyalty. This is also why the **second purchase** is the key business milestone: it moves a customer from the highest-churn group into a stickier one. (Be careful with the direction: customers who ordered more simply had more chances to be recent, so this chart shows an association, not proof that extra orders *cause* loyalty.)

# 7. Customer Segmentation (rule-based)
**Business question:** *Which groups of customers behave differently, and who is worth protecting?*


In [ ]:
# Segment 1: by BEHAVIOUR (order-count groups). Who contributes the revenue?
seg = customers.groupby("OrderGroup", observed=True).agg(
    Customers    = ("CustomerID", "count"),
    TotalRevenue = ("TotalRevenue", "sum"),
    AvgRevenue   = ("TotalRevenue", "mean"),
)
seg["Customer share %"] = seg["Customers"] / seg["Customers"].sum() * 100
seg["Revenue share %"]  = seg["TotalRevenue"] / seg["TotalRevenue"].sum() * 100
seg.round(1)

In [ ]:
# Compare "share of customers" with "share of revenue" side by side.
plot_df = seg[["Customer share %", "Revenue share %"]].reset_index().melt(id_vars="OrderGroup", var_name="Measure", value_name="Percent")

fig, ax = plt.subplots(figsize=(9, 4.5))
sns.barplot(data=plot_df, x="OrderGroup", y="Percent", hue="Measure", palette=["#9ecae1", "#08519c"], ax=ax)
for c in ax.containers: ax.bar_label(c, fmt="%.0f%%", padding=2)
ax.yaxis.set_major_formatter(PercentFormatter())
ax.set_title("Few frequent customers generate a large part of revenue", fontsize=13, weight="bold")
ax.set_xlabel(""); ax.set_ylabel("")
plt.show()

In [ ]:
# Revenue concentration (Pareto): what share of revenue do the top customers make?
sorted_rev = customers["TotalRevenue"].sort_values(ascending=False).reset_index(drop=True)
cum_share = sorted_rev.cumsum() / sorted_rev.sum() * 100                 # running total as a %
customer_pct = (np.arange(1, len(sorted_rev) + 1) / len(sorted_rev)) * 100

top20 = cum_share[int(len(sorted_rev) * 0.2) - 1]
print(f"The top 20% of customers generate {top20:.0f}% of revenue")

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(customer_pct, cum_share, linewidth=2)
ax.plot([0, 100], [0, 100], linestyle="--", color="grey", label="If every customer spent the same")
ax.axvline(20, color="red", linestyle=":")
ax.annotate(f"Top 20% -> {top20:.0f}% of revenue", (20, top20), textcoords="offset points", xytext=(15, -25), color="red")
ax.set_title("Revenue concentration", fontsize=13, weight="bold")
ax.set_xlabel("% of customers (biggest spenders first)"); ax.set_ylabel("% of revenue")
ax.legend()
plt.show()

**How to read the Pareto curve:** the further the curve bows above the dashed diagonal, the more revenue depends on a few customers. The business use: **protecting the top group matters disproportionately**.

### Segment 2: by the category of the first purchase
**Question:** *Does what a customer buys first affect whether they come back?* If yes, we could push the "sticky" categories in acquisition ads.

In [ ]:
# Find each customer's first order number
first_order_id = orders[orders["OrderNumber"] == 1][["CustomerID", "InvoiceNo"]]

# Keep only the product lines that belong to those first orders (merge keeps matching rows only)
first_lines = df.merge(first_order_id, on=["CustomerID", "InvoiceNo"], how="inner")

# Revenue per (customer, category), then keep the biggest category per customer
cat_rev = first_lines.groupby(["CustomerID", "Description"])["Revenue"].sum().reset_index()
first_cat = cat_rev.sort_values("Revenue").drop_duplicates("CustomerID", keep="last")[["CustomerID", "Description"]]
first_cat = first_cat.rename(columns={"Description": "FirstCategory"})

customers = customers.merge(first_cat, on="CustomerID", how="left")
customers["Repeater"] = (customers["Orders"] >= 2).astype(int)         # 1 = bought again, 0 = never
customers[["CustomerID", "FirstCategory", "Orders", "Repeater"]].head()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))
order = customers.groupby("FirstCategory")["Repeater"].mean().sort_values(ascending=False).index

# barplot of a 0/1 column = the share of 1s (the repeat rate), with a 95% confidence whisker
sns.barplot(data=customers, x="FirstCategory", y="Repeater", order=order, color="#4c72b0", ax=ax)
ax.axhline(customers["Repeater"].mean(), color="red", linestyle="--", label="All customers")
ax.yaxis.set_major_formatter(PercentFormatter(1.0))
ax.set_ylim(0, 1)
ax.set_title("Repeat purchase rate by first-purchase category", fontsize=13, weight="bold")
ax.set_xlabel(""); ax.set_ylabel("% who bought again")
ax.tick_params(axis="x", rotation=45)
ax.legend()
plt.show()

**Reading it:** repeat rates range from about 63% (Cloths) to 69% (Furniture), against 65.6% overall. Each category has only about 390-460 customers, and the **whiskers (95% confidence intervals) overlap heavily**, so these gaps are **within normal random variation**.

> **Finding:** the first category a customer buys does **not** reliably change whether they come back. Do **not** shift acquisition budget toward "sticky" categories on this evidence. (A negative result is still a result: it saves the business from acting on noise.)

### Segment 3: by geography and why we do **not** do it

> **Decision (log #10):** no geographic segmentation. **Recommendation to the business:** capture a delivery/billing country per customer. Knowing *when not to analyse*.

### Segment 4: by acquisition month
cohort size and month-1 retention differ by acquisition month, and we compare them in Section 5 cohort analysis (rows of the heatmap)

# 8. Machine Learning: RFM + K-Means Clustering (Unsupervised Learning)



In [ ]:
# pick the three RFM columns. Rename Orders ,Frequency and TotalRevenue , Monetary for clarity
rfm = customers[["CustomerID", "Recency", "Orders", "TotalRevenue"]].rename(columns={"Orders": "Frequency", "TotalRevenue": "Monetary"})

# Look at the shapes before we transform them
fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))
for ax, col in zip(axes, ["Recency", "Frequency", "Monetary"]):
    sns.histplot(rfm[col], bins=40, ax=ax, color="#4c72b0")
    ax.set_title(f"{col} (raw)", weight="bold")
fig.tight_layout()
plt.show()

**Why transform?** Frequency and Monetary are piled on the left with a long right tail. Left as they are, K-Means would spend its effort on a handful of extreme customers.

In [ ]:
# log transform (log1p = log(1 + x), safe when a value could be 0), then scale
rfm_log = np.log1p(rfm[["Recency", "Frequency", "Monetary"]])
scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm_log)       # a NumPy array: each column now has mean ~0 and std ~1

print("Means after scaling:", rfm_scaled.mean(axis=0).round(2))
print("Std after scaling  :", rfm_scaled.std(axis=0).round(2))

## 8.4 Choosing k - the biggest ML decision

In [ ]:
inertias, silhouettes = [], []
k_values = range(2, 9)

for k in k_values:
    model = KMeans(n_clusters=k, n_init=10, random_state=42)     # random_state = same answer every run
    labels = model.fit_predict(rfm_scaled)
    inertias.append(model.inertia_)
    silhouettes.append(silhouette_score(rfm_scaled, labels))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(list(k_values), inertias, marker="o")
axes[0].set_title("Elbow method (lower = tighter clusters)", weight="bold")
axes[0].set_xlabel("Number of clusters (k)"); axes[0].set_ylabel("Inertia")

axes[1].plot(list(k_values), silhouettes, marker="o", color="#dd8452")
axes[1].set_title("Silhouette score (higher = better separated)", weight="bold")
axes[1].set_xlabel("Number of clusters (k)"); axes[1].set_ylabel("Silhouette")
fig.tight_layout()
plt.show()

pd.DataFrame({"k": list(k_values), "inertia": np.round(inertias), "silhouette": np.round(silhouettes, 3)})

**Reading the two charts and the table**
- **Elbow:** inertia falls smoothly with no sharp bend, so the elbow does not give a clear answer.
- **Silhouette:** k=2 scores highest (0.41), k=3 and k=4 are tied at **0.31**, then it stays flat at roughly 0.28-0.30 for k=5 to 8. Scores around 0.3 mean the groups are **reasonably distinct but overlapping**, normal for customer behaviour because customers sit on a spectrum.

> **Decision (log #12) choose k = 4.** k=2 scores best but only says "active vs lapsed", which is too coarse to act on. k=3 and k=4 score the same, and **k=4 gives four groups with four different actions**.

In [ ]:
# Fit the final model with our chosen k and attach each customer's cluster number
K = 4
kmeans = KMeans(n_clusters=K, n_init=10, random_state=42)
rfm["Cluster"] = kmeans.fit_predict(rfm_scaled)

print("Silhouette score:", round(silhouette_score(rfm_scaled, rfm["Cluster"]), 3))
rfm["Cluster"].value_counts().sort_index()

In [ ]:
# PROFILE the clusters: groupby cluster and look at the averages in the ORIGINAL units (days, orders, money)
profile = rfm.groupby("Cluster").agg(
    Customers  = ("CustomerID", "count"),
    AvgRecency = ("Recency", "mean"),
    AvgOrders  = ("Frequency", "mean"),
    AvgSpend   = ("Monetary", "mean"),
    Revenue    = ("Monetary", "sum"),
)
profile["Customer share %"] = profile["Customers"] / profile["Customers"].sum() * 100
profile["Revenue share %"]  = profile["Revenue"] / profile["Revenue"].sum() * 100
profile.round(1)

**The business reading:** a third of customers are lapsed and worth only about 7% of revenue, while **18% of customers generate 58% of revenue**. That is why retention effort should be **unequal**: protect the champions, rescue the cooling regulars, nurture the promising, and spend little on the lapsed.

In [ ]:
# name each cluster by what its profile says, using simple rules:
champ    = profile["AvgSpend"].idxmax()        # spends the most           
lapsed   = profile["AvgRecency"].idxmax()      # longest since last order  
rest     = profile.drop(index=[champ, lapsed]) # the two clusters left
promising = rest["AvgSpend"].idxmin()          # smaller spend of the two   
cooling   = rest["AvgSpend"].idxmax()          # bigger spend of the two    

cluster_names = {champ: "Champions", lapsed: "Lapsed One-Timers", promising: "New & Promising", cooling: "Cooling Regulars"}
rfm["Segment"] = rfm["Cluster"].map(cluster_names)
customers = customers.merge(rfm[["CustomerID", "Cluster", "Segment"]], on="CustomerID", how="left")

seg_summary = customers.groupby("Segment").agg(
    Customers = ("CustomerID", "count"),
    AvgRecency = ("Recency", "mean"),
    AvgOrders = ("Orders", "mean"),
    AvgSpend = ("TotalRevenue", "mean"),
    ChurnRate = ("Churned", "mean"),
)
seg_summary["ChurnRate"] = seg_summary["ChurnRate"] * 100
seg_summary.round(1)

In [ ]:
# how does each segment differ on R, F and M? 
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for ax, col in zip(axes, ["Recency", "Orders", "TotalRevenue"]):
    sns.boxplot(data=customers, x="Segment", y=col, ax=ax, palette="Set2", hue="Segment", legend=False)
    ax.set_title(col, weight="bold")
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=30)
    if col != "Recency": ax.set_yscale("log")
fig.suptitle("How the segments differ", fontsize=14, weight="bold")
fig.tight_layout()
plt.show()

In [ ]:
# PCA to give a 2d view of the clusters
pca = PCA(n_components=2, random_state=42)
coords = pca.fit_transform(rfm_scaled)
print("Share of information kept by the 2 PCA axes:", pca.explained_variance_ratio_.sum().round(2))

fig, ax = plt.subplots(figsize=(8, 6))
sns.scatterplot(x=coords[:, 0], y=coords[:, 1], hue=customers["Segment"], palette="Set2", alpha=0.6, s=25, ax=ax)
ax.set_title("Customer segments seen through PCA", fontsize=13, weight="bold")
ax.set_xlabel("PCA axis 1"); ax.set_ylabel("PCA axis 2")
ax.legend(title="Segment", bbox_to_anchor=(1.02, 1), loc="upper left")   # move the legend outside the chart
plt.show()

**Reading the PCA picture:** each dot is a customer, coloured by segment. The groups form distinct regions that **touch at the edges**. That matches the silhouette score of about 0.3: customers sit on a spectrum, so the borders between segments are soft. Treat the segments as practical working groups, not hard boundaries. (Customers near an edge could easily be moved by a small change in behaviour.)

In [ ]:
# customers vs revenue per segment - who matters commercially?
share = seg_summary[["Customers"]].copy()
share["Customer share %"] = share["Customers"] / share["Customers"].sum() * 100
rev = customers.groupby("Segment")["TotalRevenue"].sum()
share["Revenue share %"] = rev / rev.sum() * 100

plot_df = share[["Customer share %", "Revenue share %"]].reset_index().melt(id_vars="Segment", var_name="Measure", value_name="Percent")
fig, ax = plt.subplots(figsize=(9, 4.5))
sns.barplot(data=plot_df, x="Segment", y="Percent", hue="Measure", palette=["#9ecae1", "#08519c"], ax=ax)
for c in ax.containers: ax.bar_label(c, fmt="%.0f%%", padding=2)
ax.yaxis.set_major_formatter(PercentFormatter())
ax.set_title("Share of customers vs share of revenue by segment", fontsize=13, weight="bold")
ax.set_xlabel(""); ax.set_ylabel("")
ax.tick_params(axis="x", rotation=15)
plt.show()

# 9. Customer Lifetime Value (CLV) Inputs
**Business question:** *How much is a customer worth in their first months, and is it worth what we spend to get them?*

**CLV = what a customer is worth over the whole relationship.** The honest way to estimate it from history is a **cohort revenue curve**: the *cumulative revenue per customer* months after joining.

In [ ]:
# Revenue per cohort per month-since-first-purchase 
rev_data  = orders_w.groupby(["CohortMonth", "CohortIndex"])["Revenue"].sum().reset_index()
rev_pivot = rev_data.pivot(index="CohortMonth", columns="CohortIndex", values="Revenue")

# cumsum(axis=1) = running total across the months; fillna(0) so blanks do not break it
cum_rev = rev_pivot.fillna(0).cumsum(axis=1).where(cohort_counts.notna()) 

# Divide by cohort size -
cum_rev_pc = cum_rev.divide(cohort_size, axis=0)

fig, ax = plt.subplots(figsize=(10, 5))
for cohort in cum_rev_pc.index:                               # one thin line per cohort
    ax.plot(cum_rev_pc.columns, cum_rev_pc.loc[cohort] / 1e3, color="lightgrey", linewidth=1)
avg_curve = cum_rev_pc.drop(index="2024-12").mean() / 1e3
ax.plot(avg_curve.index, avg_curve.values, color="#08519c", linewidth=3, marker="o", label="Average of 2025 cohorts")
ax.set_title("Cumulative revenue per customer since first purchase", fontsize=14, weight="bold")
ax.set_xlabel("Months since first purchase"); ax.set_ylabel("Revenue per customer (thousands)")
ax.legend()
plt.show()

print("Average cumulative revenue per customer (thousands), by months since first purchase:")
print(avg_curve.round(0))

In [ ]:
# The key CLV inputs in one small table
orders_per_customer = len(orders) / customers["Orders"].count()
aov = orders["Revenue"].mean()

clv_inputs = pd.Series({
    "Customers analysed": len(customers),
    "Average order value (AOV)": round(aov),
    "Orders per customer (all time in data)": round(orders_per_customer, 2),
    "Revenue per customer (all time in data)": round(customers["TotalRevenue"].mean()),
    "Repeat purchase rate %": round(customers["Repeater"].mean() * 100, 1),
    "Revenue per customer after 4 months (avg 2025 cohorts)": round(cum_rev_pc.drop(index="2024-12")[3].mean()),
})
clv_inputs

**Reading the CLV curve:** the average 2025 cohort generates about **69k per customer in the first month, about 110k after four months of activity (index 3 on the chart) and about 147k after seven months (index 6)** (the currency is not stated in the data). Every month a customer stays adds revenue, which is why retention is worth money.

**What we can and cannot say about CLV**
- We can say: *an average customer from a 2025 cohort is worth about 110k in revenue within four months.*
- We **cannot** compute profit-based CLV or the **CLV : CAC ratio**: we have no cost, margin or marketing-spend data. The rule of thumb is that CLV should be at least 3x CAC. Once marketing supplies cost per new customer, the calculation is `CAC = marketing spend / new customers` and `ratio = CLV / CAC`.

# 10. Key Numbers Behind the Findings
All the figures quoted in the summary below are printed here, so you can always trace a claim back to code.

In [ ]:
avg2025 = retention.drop(index="2024-12")
print("Customers analysed:", len(customers), "| orders:", len(orders))
print("Repeat purchase rate:", round(customers["Repeater"].mean() * 100, 1), "%   One-time buyers:", round((customers["Orders"] == 1).mean() * 100, 1), "%")
print("Average month-1 retention (2025 cohorts):", round(avg2025[1].mean(), 1), "%")
print("Average month-2 / 3 / 6 retention:", [round(avg2025[i].mean(), 1) for i in [2, 3, 6]])
print("Dec-2024 cohort month-1 retention:", round(retention.loc["2024-12", 1], 1), "%")
print("Month-1 retention range across 2025 cohorts:", round(avg2025[1].min(), 1), "to", round(avg2025[1].max(), 1))
print("Best / worst month-1 cohort:", avg2025[1].idxmax(), "/", avg2025[1].idxmin())
print("Churn threshold (days):", churn_days, "| churned share:", round(customers["Churned"].mean() * 100, 1), "%")
print("Second order within 30/60/90/180 days:", [round(cum_pct[d], 1) for d in [30, 60, 90, 180]])
print("Top 20% of customers = ", round(top20), "% of revenue")
print("Churn rate by order group:", churn_by_group.round(1).to_dict())
print("Repeat rate by first category (min / max):", customers.groupby("FirstCategory")["Repeater"].mean().round(3).agg(["min", "max"]).to_dict())
print("Avg new customers per month 2025 (Jan-Aug / Sep-Nov):", round(cohort_size.loc["2025-01":"2025-08"].mean()), "/", round(cohort_size.loc["2025-09":"2025-11"].mean()))

# Strategic Recommendations & KPIs

| # | Recommendation | Evidence from this notebook | KPI (baseline from the data) | Suggested target |
|---|---|---|---|---|
| 1 | **New-customer journey:** welcome email, a reason to return, and a reminder with an incentive between **day 14 and day 45**. | Only 18% reorder in 30 days; 33% in 60; the median second order is at about day 71, so a reminder at day 14-45 arrives before most customers reorder. | **Second order within 60 days:** 33%. **Month-1 retention:** 20% | +5 points each |
| 2 | **Win-back trigger for regulars** at **about 75-100 days** of silence (before the 102-day churn line). | Cooling Regulars average 78 days since last order and 23% are already churned. | **Churn rate of regulars:** 23%. **Reactivation rate** after the trigger | Churn down by a third |
| 3 | **Protect Champions:** early access, loyalty tiers, a named contact. | 18% of customers bring 58% of revenue; the top 20% bring 66%. | **Champion churn:** about 0%. **Share of customers who are Champions:** 18% | Keep churn near 0, grow the share |
| 4 | **Keep spend on one-time buyers low:** automated emails only, no heavy discounts. | Lapsed One-Timers: 34% of customers, 7% of revenue, 72% churned. | **Cost per reactivated customer** vs revenue they return | Cost below one order's value |
| 5 | **Nurture "New & Promising" to a third order.** | Continuation rates rise from 66% to 71% to 75% as orders build. | **% of 2-order customers who place a 3rd:** 71% | 75%+ |
| 6 | **Judge acquisition on cohort quality, not only volume.** | Autumn brought more customers *and* better month-1 retention (23-24%); summer brought fewer and weaker. | **Month-1 retention by channel and month** | Report monthly |
| 7 | **Fix the data.** Capture a customer ID on every order, one country per customer, acquisition channel, marketing cost, margin, and proper cancellation records. | 25% of rows had no customer; countries were inconsistent; no cost data. | **% of order lines with a customer ID:** 75% | 95%+ |
| 8 | **Test before scaling.** Run each idea as an A/B test on a slice of customers and compare cohorts. | Correlation here is not proof of cause. | Retention of test vs control | Statistically clear uplift |

**Not recommended on this evidence:** shifting acquisition toward "sticky" first-purchase categories (differences were within chance), or country-based campaigns (the data cannot support them).